In [1]:
import os

In [2]:
%pwd

'/home/vk/Desktop/Python_Code/Pytorch/NLP/Text_Summarizer_Project/research'

In [3]:
os.chdir("../")
%pwd

'/home/vk/Desktop/Python_Code/Pytorch/NLP/Text_Summarizer_Project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path : Path 
    tokenizer_name: Path


In [5]:
from textSummarizer.constants import * 
from textSummarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
            
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])

        
    def get_data_transformation_config(self)-> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir= config.root_dir,
            data_path= config.data_path,
            tokenizer_name= config.tokenizer_name
            )
        
        return data_transformation_config


In [7]:
import os 
from textSummarizer.logging import logging
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[2026-06-02 01:40:02,334: INFO: utils: Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.]
[2026-06-02 01:40:02,334: INFO: utils: NumExpr defaulting to 16 threads.]


In [10]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_example_to_features(self, example_batch):
        input_encoding = self.tokenizer(example_batch['dialogue'], max_length = 1024, truncation = True )
    
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True
        )

        return {
            'input_ids' : input_encoding['input_ids'],
            'attention_mask': input_encoding['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_example_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))
        

In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e    

[2026-06-02 01:49:32,691: INFO: common: yaml file: config/config.yaml loaded successfully ]
[2026-06-02 01:49:32,693: INFO: common: yaml file: params.yaml loaded successfully ]
[2026-06-02 01:49:32,694: INFO: common: created directory at: artifacts]
[2026-06-02 01:49:32,694: INFO: common: created directory at: artifacts/data_transformation]
[2026-06-02 01:49:32,961: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-02 01:49:32,978: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-06-02 01:49:33,238: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-02 01:49:33,254: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]